# QM9 Dataset: Loading, EDA, Interpretation and Preprocessing

This notebook has four parts:

1. **Load** the QM9 dataset (about 134k small organic molecules, up to 9 heavy atoms: C, N, O, F).
2. **EDA**: data quality, target distributions, correlations, molecular composition, size effects, outliers.
3. **Interpret** the EDA: what each pattern means for modelling.
4. **Preprocess**: clean, choose a leak-free target, build features, split, scale, and (optionally) prepare inputs for a quantum circuit.

Run the cells from top to bottom. Change the settings in the **Configuration** cell to control the target, sample size and qubit count.

## 0. Setup

In [3]:
# Run once if packages are missing
!pip install pandas numpy matplotlib seaborn scikit-learn rdkit joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.3/30.3 MB 3.3 MB/s  0:00:09 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
np.random.seed(42)

### Configuration

In [5]:
DATA_PATH   = "qm9.csv"   # local copy; downloaded automatically if missing
DATA_URL    = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/qm9.csv"
UNCHAR_URL  = "https://ndownloader.figshare.com/files/3195404"  # list of ~3,054 molecules that failed QM9's consistency check

TARGET      = "gap_eV"    # options: gap_eV, homo_eV, lumo_eV, mu, alpha, cv, zpve, r2, u0_atom_eV
SAMPLE_SIZE = None        # e.g. 5000 for a quick run or for quantum models; None = full dataset
SPLIT       = "random"    # "random" or "scaffold"
N_QUBITS    = 8           # number of features kept for quantum angle encoding
OUT_DIR     = "processed"
RANDOM_STATE = 42

## 1. Loading the dataset

We use the CSV version of QM9 from MoleculeNet/DeepChem. Each row is one molecule with its SMILES string and the computed properties.

| Column | Meaning | Unit in file | Type |
|---|---|---|---|
| `A`, `B`, `C` | Rotational constants | GHz | – |
| `mu` | Dipole moment | Debye | Intensive |
| `alpha` | Isotropic polarizability | Bohr³ | Extensive |
| `homo`, `lumo`, `gap` | Orbital energies and HOMO–LUMO gap | Hartree | Intensive |
| `r2` | Electronic spatial extent | Bohr² | Extensive |
| `zpve` | Zero-point vibrational energy | Hartree | Extensive |
| `u0`, `u298`, `h298`, `g298` | Internal energy (0 K, 298 K), enthalpy, free energy | Hartree | Extensive |
| `cv` | Heat capacity at 298 K | cal/(mol·K) | Extensive |

**Intensive** properties do not grow with molecule size. **Extensive** properties do.

> If the download is blocked on your network, download `qm9.csv` manually from the URL above and put it next to this notebook.

In [6]:
if os.path.exists(DATA_PATH):
    df_raw = pd.read_csv(DATA_PATH)
else:
    df_raw = pd.read_csv(DATA_URL)
    df_raw.to_csv(DATA_PATH, index=False)   # cache for next time

print("Shape:", df_raw.shape)
df_raw.head()

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018)>

In [ ]:
TARGETS = ["mu", "alpha", "homo", "lumo", "gap", "r2", "zpve", "u0", "u298", "h298", "g298", "cv"]
TARGETS = [t for t in TARGETS if t in df_raw.columns]
print("Columns:", list(df_raw.columns))
print("Targets used in EDA:", TARGETS)

## 2. Exploratory Data Analysis

### 2.1 Structure and data quality

In [ ]:
df = df_raw.copy()
df.info()

In [ ]:
quality = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(3),
    "n_unique": df.nunique(),
})
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate SMILES:", df["smiles"].duplicated().sum())
quality

In [ ]:
# Parse SMILES with RDKit; invalid strings become None
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)
print("Invalid SMILES:", df["mol"].isna().sum())

### 2.2 Summary statistics and skewness

In [ ]:
summary = df[TARGETS].describe().T
summary["skew"] = df[TARGETS].skew()
summary["kurtosis"] = df[TARGETS].kurt()
summary.round(4)

### 2.3 Target distributions

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 11))
for ax, t in zip(axes.ravel(), TARGETS):
    sns.histplot(df[t], bins=60, kde=True, ax=ax, color="#3b6ea5")
    ax.set_title(f"{t}  (skew={df[t].skew():.2f})")
    ax.set_xlabel("")
for ax in axes.ravel()[len(TARGETS):]:
    ax.axis("off")
plt.suptitle("Distribution of QM9 targets", y=1.01, fontsize=15)
plt.tight_layout(); plt.show()

### 2.4 Correlations between targets

In [ ]:
corr = df[TARGETS].corr()
plt.figure(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Pearson correlation between targets"); plt.show()

# Highly correlated pairs
pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack().rename("r").reset_index())
pairs["|r|"] = pairs["r"].abs()
pairs.sort_values("|r|", ascending=False).head(12)

In [ ]:
# Check the identity gap = lumo - homo
diff = (df["lumo"] - df["homo"] - df["gap"]).abs()
print(f"max |lumo - homo - gap| = {diff.max():.2e} Hartree")

### 2.5 Molecular composition

In [ ]:
def composition(m):
    mh = Chem.AddHs(m)
    sym = [a.GetSymbol() for a in mh.GetAtoms()]
    return pd.Series({
        "n_atoms": len(sym),
        "n_heavy": m.GetNumHeavyAtoms(),
        "n_H": sym.count("H"), "n_C": sym.count("C"), "n_N": sym.count("N"),
        "n_O": sym.count("O"), "n_F": sym.count("F"),
        "n_rings": rdMolDescriptors.CalcNumRings(m),
        "n_arom_rings": rdMolDescriptors.CalcNumAromaticRings(m),
        "mol_wt": Descriptors.MolWt(m),
    })

valid = df["mol"].notna()
comp = df.loc[valid, "mol"].apply(composition)
df = df.join(comp)
comp.describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
df["n_heavy"].value_counts().sort_index().plot.bar(ax=axes[0], color="#3b6ea5")
axes[0].set_title("Molecules by number of heavy atoms"); axes[0].set_xlabel("heavy atoms")

contains = {el: (df[f"n_{el}"] > 0).mean() * 100 for el in ["C", "N", "O", "F"]}
pd.Series(contains).plot.bar(ax=axes[1], color="#e07b39")
axes[1].set_title("% of molecules containing each element"); axes[1].set_ylabel("%")

df["n_rings"].value_counts().sort_index().plot.bar(ax=axes[2], color="#5a9e6f")
axes[2].set_title("Molecules by number of rings"); axes[2].set_xlabel("rings")
plt.tight_layout(); plt.show()

### 2.6 How molecule size relates to the targets

In [ ]:
sample = df.sample(min(8000, len(df)), random_state=RANDOM_STATE)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.scatterplot(data=sample, x="n_atoms", y="u0", s=8, alpha=0.4, ax=axes[0])
axes[0].set_title("u0 vs total atoms (extensive)")
sns.boxplot(data=df, x="n_heavy", y="gap", ax=axes[1], color="#9ecae1", fliersize=1)
axes[1].set_title("gap vs heavy atoms (intensive)")
sns.boxplot(data=df, x="n_heavy", y="alpha", ax=axes[2], color="#fdae6b", fliersize=1)
axes[2].set_title("alpha vs heavy atoms (extensive)")
plt.tight_layout(); plt.show()

size_corr = df[TARGETS + ["n_atoms", "n_heavy"]].corr()[["n_atoms", "n_heavy"]].loc[TARGETS]
size_corr.round(3)

In [ ]:
# How much of u0 can be explained just by counting atoms of each element?
from sklearn.linear_model import LinearRegression
Xc = df.loc[valid, ["n_H", "n_C", "n_N", "n_O", "n_F"]]
lr = LinearRegression().fit(Xc, df.loc[valid, "u0"])
print(f"R² of u0 predicted only from element counts: {lr.score(Xc, df.loc[valid, 'u0']):.5f}")

### 2.7 Outliers

In [ ]:
z = (df[TARGETS] - df[TARGETS].mean()) / df[TARGETS].std()
q1, q3 = df[TARGETS].quantile(0.25), df[TARGETS].quantile(0.75)
iqr = q3 - q1
iqr_out = ((df[TARGETS] < q1 - 1.5 * iqr) | (df[TARGETS] > q3 + 1.5 * iqr)).sum()
outliers = pd.DataFrame({"|z|>4": (z.abs() > 4).sum(), "IQR outliers": iqr_out,
                         "IQR %": (iqr_out / len(df) * 100).round(2)})
outliers

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4.5))
sns.boxplot(data=z, ax=ax, fliersize=1)
ax.set_title("Standardised targets (z-scores)"); plt.show()

# Inspect the most extreme molecules for one target
df.loc[z["mu"].abs().sort_values(ascending=False).index[:5], ["smiles", "mu", "n_heavy"]]

### 2.8 Automatic EDA summary

In [ ]:
findings = []
findings.append(f"Rows: {len(df):,}; missing values: {int(df[TARGETS].isna().sum().sum())}; "
                f"duplicate SMILES: {df['smiles'].duplicated().sum()}; invalid SMILES: {int(df['mol'].isna().sum())}")
skewed = summary["skew"][summary["skew"].abs() > 1].round(2).to_dict()
findings.append(f"Strongly skewed targets (|skew| > 1): {skewed}")
near_dup = pairs[pairs['|r|'] > 0.98][["level_0", "level_1", "r"]].round(4).values.tolist()
findings.append(f"Near-duplicate target pairs (|r| > 0.98): {near_dup}")
findings.append(f"Max |lumo - homo - gap|: {diff.max():.1e} -> gap is derived from homo and lumo")
findings.append(f"u0 explained by element counts alone: R² = {lr.score(Xc, df.loc[valid, 'u0']):.5f}")
findings.append(f"Molecules with fluorine: {contains['F']:.2f}%  (rare class)")
for f in findings:
    print("•", f)

## 3. Interpreting the EDA

Compare these points with your outputs above.

1. **The data is clean.** There are no missing values and almost no invalid or duplicate SMILES. The one known issue is that about 3,054 molecules failed the original geometry-consistency check (the molecule built from the 3D geometry does not match the SMILES). These are usually removed (step 4.1).

2. **u0, u298, h298 and g298 are almost perfectly correlated (r ≈ 1.0).** They differ only by small thermal corrections, so they carry almost the same information. Choose one (usually U0), or treat them as separate tasks in multi-task learning.

3. **Total energies are driven by molecule size.** The linear model in 2.6 explains almost all of u0's variance from element counts alone. A model can therefore look very accurate on u0 without learning any chemistry. The fix is to predict **atomization energy** (u0 minus the sum of the isolated-atom energies) or energy per atom (step 4.2).

4. **gap = lumo − homo exactly.** Never use homo and lumo as input features when the target is gap (or the reverse). That is target leakage.

5. **Intensive and extensive targets behave differently.** gap, homo, lumo and mu change little with size. alpha, r2, zpve, cv and the energies rise steadily with the number of atoms. Size descriptors will be strong features for the extensive targets.

6. **Skewed targets.** `mu` is right-skewed and has many values near 0 (symmetric molecules have no dipole). `r2` and `alpha` also have long tails. A log1p transform plus standardisation helps models that assume roughly Gaussian errors.

7. **Different units and scales.** Energies are in Hartree, mu in Debye, alpha in Bohr³, and so on. Always scale the targets, especially when you train on several at once.

8. **Imbalanced chemistry.** Most molecules have 9 heavy atoms and contain carbon. Fluorine molecules are rare, so expect larger errors on them. Report per-group errors, or use a scaffold split to test generalisation.

9. **Most outliers are real chemistry,** such as strained rings or very polar molecules, not data errors. Do not drop them blindly. Keep them unless you can confirm they are wrong (for example, the uncharacterised list).

## 4. Preprocessing

### 4.1 Cleaning

In [ ]:
n0 = len(df)
df = df[df["mol"].notna()].copy()

# Canonical SMILES for reliable duplicate detection
df["can_smiles"] = df["mol"].apply(Chem.MolToSmiles)
df = df.drop_duplicates("can_smiles")

# Remove the ~3,054 'uncharacterized' molecules (index = gdb number)
try:
    import urllib.request
    lines = urllib.request.urlopen(UNCHAR_URL, timeout=30).read().decode().splitlines()
    unchar = {int(l.split()[0]) for l in lines[9:-2] if l.strip()}
    gdb_idx = df["mol_id"].str.replace("gdb_", "").astype(int)
    df = df[~gdb_idx.isin(unchar)]
    print(f"Removed {len(unchar)} uncharacterized molecules")
except Exception as e:
    print("Could not download the uncharacterized list, skipping this step:", e)

df = df.reset_index(drop=True)
print(f"Rows: {n0:,} -> {len(df):,}")

### 4.2 Unit conversion and leak-free targets

- Convert orbital energies from Hartree to eV (easier to read).
- Build the **atomization energy** from u0 using the B3LYP/6-31G(2df,p) reference energies of the isolated atoms, as published with QM9.

In [ ]:
HARTREE_TO_EV = 27.211386
ATOM_REF_U0 = {"H": -0.500273, "C": -37.846772, "N": -54.583861, "O": -75.064579, "F": -99.718730}  # Hartree

for t in ["homo", "lumo", "gap"]:
    df[f"{t}_eV"] = df[t] * HARTREE_TO_EV

atom_ref = sum(df[f"n_{el}"] * e for el, e in ATOM_REF_U0.items())
df["u0_atom_eV"] = (df["u0"] - atom_ref) * HARTREE_TO_EV
df["u0_atom_per_atom_eV"] = df["u0_atom_eV"] / df["n_atoms"]

print(df[["gap_eV", "u0_atom_eV", "u0_atom_per_atom_eV"]].describe().round(3))
print("corr(u0_atom_per_atom, n_atoms) =", round(df["u0_atom_per_atom_eV"].corr(df["n_atoms"]), 3),
      " vs corr(u0, n_atoms) =", round(df["u0"].corr(df["n_atoms"]), 3))

In [ ]:
# Columns that must NOT be used as features for the chosen target (leakage)
ALL_PROPS = ["A", "B", "C"] + TARGETS + ["homo_eV", "lumo_eV", "gap_eV",
             "u0_atom_eV", "u0_atom_per_atom_eV"] + [c for c in df.columns if c.endswith("_atom")]
LEAKY = [c for c in ALL_PROPS if c in df.columns]
print("Target:", TARGET)
print("Excluded from features:", LEAKY)

### 4.3 Optional subsample (recommended for quantum models)

In [ ]:
if SAMPLE_SIZE and SAMPLE_SIZE < len(df):
    # Stratify by heavy-atom count so the subsample keeps the size distribution
    df, _ = train_test_split(df, train_size=SAMPLE_SIZE, stratify=df["n_heavy"].clip(lower=3),
                             random_state=RANDOM_STATE)
    df = df.reset_index(drop=True)
print("Working rows:", len(df))

### 4.4 Feature engineering

Two feature sets from SMILES:

- **RDKit descriptors**: physically meaningful numbers (size, polarity, rings, charge distribution). These are good for classical ML, and they compress well with PCA for quantum circuits.
- **Morgan fingerprints** (radius 2, 1024 bits): substructure presence. Use these with tree models or neural nets.

For 3D-aware models, use the Coulomb matrix or SOAP from the xyz geometries, or graph models such as SchNet in PyTorch Geometric.

In [ ]:
DESC_NAMES = [n for n, _ in Descriptors.descList]   # ~200 descriptors
desc_fns = dict(Descriptors.descList)

def rdkit_descriptors(m):
    out = []
    for n in DESC_NAMES:
        try:
            out.append(desc_fns[n](m))
        except Exception:
            out.append(np.nan)
    return out

X_desc = pd.DataFrame([rdkit_descriptors(m) for m in df["mol"]], columns=DESC_NAMES)
count_cols = ["n_atoms", "n_heavy", "n_H", "n_C", "n_N", "n_O", "n_F", "n_rings", "n_arom_rings"]
X_desc = pd.concat([X_desc, df[count_cols].reset_index(drop=True)], axis=1)
print("Descriptor matrix:", X_desc.shape)

In [ ]:
def morgan_fp(m, radius=2, n_bits=1024):
    fp = AllChem.GetMorganFingerprintAsBitVect(m, radius, nBits=n_bits)
    return np.array(fp, dtype=np.uint8)

X_fp = np.vstack([morgan_fp(m) for m in df["mol"]])
print("Fingerprint matrix:", X_fp.shape, "| bits never set:", int((X_fp.sum(0) == 0).sum()))

### 4.5 Feature cleaning

In [ ]:
X = X_desc.replace([np.inf, -np.inf], np.nan)

# 1) Drop columns with many missing values, fill the rest with the median
X = X.loc[:, X.isna().mean() < 0.05]
X = X.fillna(X.median())

# 2) Drop constant / near-constant columns
X = X.loc[:, X.std() > 1e-6]

# 3) Drop one of each pair of highly correlated features (|r| > 0.95)
cm = X.corr().abs()
upper = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool))
drop_corr = [c for c in upper.columns if (upper[c] > 0.95).any()]
X = X.drop(columns=drop_corr)

print(f"Features: {X_desc.shape[1]} -> {X.shape[1]}  (dropped {len(drop_corr)} highly correlated)")

### 4.6 Train / validation / test split (before any fitting)

In [ ]:
y = df[TARGET].values
idx = np.arange(len(df))

if SPLIT == "scaffold":
    # Molecules sharing a Bemis–Murcko scaffold go to the same split: a harder, more realistic test
    scaff = df["mol"].apply(lambda m: MurckoScaffold.MurckoScaffoldSmiles(mol=m))
    groups = scaff.groupby(scaff).groups
    order = sorted(groups.values(), key=len, reverse=True)
    tr, va, te = [], [], []
    n = len(df)
    for g in order:
        g = list(g)
        if len(tr) + len(g) <= 0.8 * n: tr += g
        elif len(va) + len(g) <= 0.1 * n: va += g
        else: te += g
    idx_tr, idx_va, idx_te = map(np.array, (tr, va, te))
else:
    idx_tr, idx_te = train_test_split(idx, test_size=0.10, random_state=RANDOM_STATE)
    idx_tr, idx_va = train_test_split(idx_tr, test_size=1/9, random_state=RANDOM_STATE)

X_tr, X_va, X_te = X.values[idx_tr], X.values[idx_va], X.values[idx_te]
y_tr, y_va, y_te = y[idx_tr], y[idx_va], y[idx_te]
print(f"Split = {SPLIT}: train {len(idx_tr):,} | val {len(idx_va):,} | test {len(idx_te):,}")

### 4.7 Scaling (fit on the training set only)

- **Features**: StandardScaler.
- **Target**: log1p first if it is skewed and non-negative (for example mu), then StandardScaler. Keep the scaler so predictions can be converted back to real units.

In [ ]:
x_scaler = StandardScaler().fit(X_tr)
X_tr_s, X_va_s, X_te_s = (x_scaler.transform(a) for a in (X_tr, X_va, X_te))

LOG_TARGET = (pd.Series(y_tr).skew() > 1) and (y_tr.min() >= 0)
f_y = np.log1p if LOG_TARGET else (lambda v: v)
y_scaler = StandardScaler().fit(f_y(y_tr).reshape(-1, 1))
y_tr_s, y_va_s, y_te_s = (y_scaler.transform(f_y(a).reshape(-1, 1)).ravel() for a in (y_tr, y_va, y_te))

def inverse_target(y_scaled):
    v = y_scaler.inverse_transform(np.asarray(y_scaled).reshape(-1, 1)).ravel()
    return np.expm1(v) if LOG_TARGET else v

print("log1p applied to target:", LOG_TARGET)
print("Train features mean≈0, std≈1:", X_tr_s.mean().round(3), X_tr_s.std().round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
sns.histplot(y_tr, bins=60, ax=axes[0]); axes[0].set_title(f"{TARGET} (raw)")
sns.histplot(y_tr_s, bins=60, ax=axes[1], color="#e07b39"); axes[1].set_title(f"{TARGET} (transformed)")
plt.tight_layout(); plt.show()

### 4.8 Quantum ML preparation: reduce dimensions and encode as angles

A quantum circuit with angle encoding uses one rotation per qubit, so we:

1. Reduce the scaled descriptors to `N_QUBITS` components with PCA (fit on train).
2. Rescale each component to **[0, π]** so each value maps to a rotation such as RY(θ).

In [ ]:
pca = PCA(n_components=N_QUBITS, random_state=RANDOM_STATE).fit(X_tr_s)
P_tr, P_va, P_te = (pca.transform(a) for a in (X_tr_s, X_va_s, X_te_s))

angle_scaler = MinMaxScaler(feature_range=(0, np.pi)).fit(P_tr)
Q_tr, Q_va, Q_te = (np.clip(angle_scaler.transform(a), 0, np.pi) for a in (P_tr, P_va, P_te))

evr = pca.explained_variance_ratio_
print(f"Variance explained by {N_QUBITS} components: {evr.sum()*100:.1f}%")
plt.figure(figsize=(8, 3.5))
plt.bar(range(1, N_QUBITS + 1), evr * 100, label="per component")
plt.plot(range(1, N_QUBITS + 1), np.cumsum(evr) * 100, "o-", color="#e07b39", label="cumulative")
plt.xlabel("principal component (qubit)"); plt.ylabel("% variance"); plt.legend()
plt.title("PCA for qubit encoding"); plt.show()
print("Angle range in train:", Q_tr.min().round(3), "to", Q_tr.max().round(3))

### 4.9 Save the processed data

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
np.savez_compressed(
    os.path.join(OUT_DIR, f"qm9_{TARGET}_processed.npz"),
    X_tr=X_tr_s, X_va=X_va_s, X_te=X_te_s,
    y_tr=y_tr_s, y_va=y_va_s, y_te=y_te_s,
    Q_tr=Q_tr, Q_va=Q_va, Q_te=Q_te,
    fp_tr=X_fp[idx_tr], fp_va=X_fp[idx_va], fp_te=X_fp[idx_te],
    idx_tr=idx_tr, idx_va=idx_va, idx_te=idx_te,
    feature_names=np.array(X.columns),
)
joblib.dump({"x_scaler": x_scaler, "y_scaler": y_scaler, "log_target": LOG_TARGET,
             "pca": pca, "angle_scaler": angle_scaler, "features": list(X.columns), "target": TARGET},
            os.path.join(OUT_DIR, f"qm9_{TARGET}_transforms.joblib"))
df.drop(columns=["mol"]).to_csv(os.path.join(OUT_DIR, "qm9_clean.csv"), index=False)
print("Saved to", os.path.abspath(OUT_DIR))

## 5. Preprocessing checklist

- [x] Removed invalid SMILES, duplicates and the uncharacterised molecules
- [x] Chose a leak-free target (atomization energy instead of raw u0; homo/lumo excluded when predicting gap)
- [x] Built RDKit descriptors and Morgan fingerprints
- [x] Removed constant and highly correlated features
- [x] Split **before** fitting any scaler or PCA (random or scaffold)
- [x] Scaled features and target on the training set only (log1p for skewed targets)
- [x] PCA to `N_QUBITS` and scaled to [0, π] for quantum angle encoding
- [x] Saved arrays and fitted transforms for the modelling notebook

**Next step:** train a classical baseline (for example Random Forest or Kernel Ridge on `X_tr`) and a quantum model (for example a VQR or quantum kernel on `Q_tr`) on the same split, and compare MAE in real units with `inverse_target()`.